# Temporal model experiments: recall at a precision floor

This notebook preserves the project's task definitions and evaluation protocol:

- **Model A:** attempt-level classification for `target_model_a_dropout_after_next_attempt`.
- **Model B:** student-level classification for `target_model_b_eventual_dropout`. A student is flagged if **any eligible landmark** has a predicted probability at or above the threshold. For threshold selection, probabilities are aggregated as the maximum across that student's eligible landmarks.
- Split students, not rows. Use validation predictions to select the threshold that maximizes recall subject to precision ≥ 0.40, 0.50, or 0.60.
- Report test precision/recall and whether the test precision floor was met. Do not select thresholds on test data.

**Caveat:** this is a retrospective student-cohort split, not a fully prospective event-time validation. The supplied dropout label has no event timestamp. Treat the results as experiments on the supplied labels, not proof of future dropout prediction.

Outputs are written into a dedicated subfolder under `outputs/`, with separate `metrics/` and `predictions/` directories.

## 1. Load data and establish output folders

In [1]:
from pathlib import Path
import warnings, json, sys
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
from IPython.display import display

ROOT_CANDIDATES = [Path("..").resolve(), Path("/mnt/data/ssassistant_next/student_success_assistant").resolve(),
                   Path("/mnt/data/student_success_assistant").resolve(), Path(".").resolve()]
ROOT = next((p for p in ROOT_CANDIDATES if (p / "data" / "curated" / "temporal_attempt_prediction_rows.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate data/curated/temporal_attempt_prediction_rows.csv. Run the temporal feature-engineering notebook first.")

DATA = ROOT / "data" / "curated"
EXPERIMENT_NAME = "temporal_small_mlp"
OUT = ROOT / "outputs" / EXPERIMENT_NAME
METRICS_OUT = OUT / "metrics"
PRED_OUT = OUT / "predictions"
for folder in (METRICS_OUT, PRED_OUT):
    folder.mkdir(parents=True, exist_ok=True)

samples = pd.read_csv(DATA / "temporal_attempt_prediction_rows.csv",
                      parse_dates=["attempt_dt", "signup_dt", "student_last_attempt_dt"])
print("Project root:", ROOT)
print("Output root:", OUT)
print("Prediction rows:", len(samples), "| students:", samples.student_id.nunique())
print("Model A positives:", int(samples.target_model_a_dropout_after_next_attempt.sum()))
print("Model B positive rate:", round(samples.groupby("student_id").target_model_b_eventual_dropout.first().mean(), 3))

Project root: /Users/abishek/Documents/Projects/student_success_assistant
Output root: /Users/abishek/Documents/Projects/student_success_assistant/outputs/temporal_small_mlp
Prediction rows: 1476 | students: 438
Model A positives: 80
Model B positive rate: 0.183


## 2. Feature contract and ablations

Identifiers, target columns, total observed attempts, and final-observation metadata are excluded from predictors. The experiment compares history-only, static-only, and combined feature sets.

In [2]:
TARGETS = {
    "model_a_near_term_dropout": "target_model_a_dropout_after_next_attempt",
    "model_b_eventual_dropout": "target_model_b_eventual_dropout",
}
META_COLS = {
    "student_id", "attempt_id", "attempt_dt", "signup_dt", "student_last_attempt_dt",
    "observed_attempt_count", "target_dropout", "next_attempt_is_final_observed",
    "target_model_a_dropout_after_next_attempt", "target_model_b_eventual_dropout",
}
all_predictor_cols = [c for c in samples.columns if c not in META_COLS]
assert not any("target" in c.lower() for c in all_predictor_cols)
assert not ({"student_id", "attempt_id", "student_last_attempt_dt", "observed_attempt_count"} & set(all_predictor_cols))
STATIC_FEATURES = ["age_group", "country", "device", "signup_month", "signup_day_of_week", "signup_day_of_month"]
STATIC_FEATURES = [c for c in STATIC_FEATURES if c in all_predictor_cols]
TEMPORAL_FEATURES = [c for c in all_predictor_cols if c not in STATIC_FEATURES]
FEATURE_SETS = {
    "temporal_history_only": TEMPORAL_FEATURES,
    "student_static_only": STATIC_FEATURES,
    "combined_static_and_history": all_predictor_cols,
}
display(pd.DataFrame([{"feature_set": k, "feature_count": len(v), "features": ", ".join(v)}
                      for k, v in FEATURE_SETS.items()]))

,feature_set,feature_count,features
0,temporal_history_only,27,"sequence_index, days_from_signup, gap_since_pr..."
1,student_static_only,6,"age_group, country, device, signup_month, sign..."
2,combined_static_and_history,33,"sequence_index, days_from_signup, gap_since_pr..."


## 3. Split and metric utilities

The threshold is selected on validation data to maximize recall subject to the precision floor. Model B is aggregated to one maximum-risk score per student before threshold selection and evaluation.

In [3]:
def student_split(df, regime="random", seed=42):
    # Split by student-level supplied label; never split rows from one student across partitions.
    ids = df.groupby("student_id", as_index=False).agg(
        target=("target_dropout", "first"),
        first_landmark=("attempt_dt", "min")
    )
    if regime == "random":
        tr_ids, rem_ids = train_test_split(ids.student_id, test_size=0.40, random_state=seed,
                                           stratify=ids["target"])
        rem = ids[ids.student_id.isin(rem_ids)]
        va_ids, te_ids = train_test_split(rem.student_id, test_size=0.50, random_state=seed,
                                          stratify=rem["target"])
        return set(tr_ids), set(va_ids), set(te_ids), {}
    if regime == "chronological":
        ordered = ids.sort_values(["first_landmark", "student_id"]).reset_index(drop=True)
        n = len(ordered)
        a, b = int(0.60*n), int(0.80*n)
        return (set(ordered.iloc[:a].student_id), set(ordered.iloc[a:b].student_id),
                set(ordered.iloc[b:].student_id),
                {"train_cutoff": str(ordered.iloc[a-1].first_landmark.date()),
                 "validation_cutoff": str(ordered.iloc[b-1].first_landmark.date()),
                 "test_start": str(ordered.iloc[b].first_landmark.date())})
    raise ValueError(regime)

def binary_metrics(y, prob, threshold=0.5):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(prob) >= threshold
    cm = confusion_matrix(y, pred, labels=[0, 1])
    return {
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "PR_AUC": average_precision_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        "ROC_AUC": roc_auc_score(y, prob) if len(np.unique(y)) > 1 else np.nan,
        "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
        "flagged": int(pred.sum()), "n": len(y),
    }

def choose_threshold(y, prob, precision_floor):
    # Choose maximum validation recall subject to precision >= floor.
    # If no threshold satisfies the floor, return infeasible rather than silently relaxing it.
    candidates = np.unique(np.r_[np.arange(0.01, 1.001, 0.01), 1.01])
    feasible = []
    for t in candidates:
        m = binary_metrics(y, prob, t)
        if m["precision"] >= precision_floor:
            feasible.append((m["recall"], m["precision"], t))
    if not feasible:
        return np.nan, False
    best = max(feasible, key=lambda x: (x[0], x[1], x[2]))
    return float(best[2]), True

def equal_student_resample(train_rows, seed=42):
    # MLPClassifier versions differ in sample_weight support. Resample landmarks with
    # probability inversely proportional to each student's row count so each student
    # contributes approximately equal expected training mass.
    counts = train_rows.groupby("student_id")["student_id"].transform("size").to_numpy(dtype=float)
    p = 1.0 / counts
    p /= p.sum()
    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(len(train_rows)), size=len(train_rows), replace=True, p=p)
    return train_rows.iloc[idx].copy()

## 4. Model configurations

In [4]:
from sklearn.neural_network import MLPClassifier

# Compact MLPs only; with this cohort size, larger networks are unlikely to be justified.
MODEL_CONFIGS = {
    "mlp_16": {"hidden_layer_sizes": (16,), "alpha": 0.01, "learning_rate_init": 0.001,
               "max_iter": 400, "equal_student_resample": True},
    "mlp_32_16": {"hidden_layer_sizes": (32, 16), "alpha": 0.03, "learning_rate_init": 0.0007,
                  "max_iter": 500, "equal_student_resample": True},
}

In [5]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_model(train_df, feature_cols, params):
    cat = [c for c in feature_cols if c in train_df.columns and
           (train_df[c].dtype == "object" or str(train_df[c].dtype).startswith("category"))]
    num = [c for c in feature_cols if c not in cat]
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", make_ohe()), ("scale", StandardScaler())]), cat)
    ], sparse_threshold=0.0)
    clf = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"], activation="relu",
        alpha=params["alpha"], learning_rate_init=params["learning_rate_init"],
        max_iter=params["max_iter"], early_stopping=False,
        random_state=42)
    return Pipeline([("prep", prep), ("model", clf)])

## 5. Fit, select thresholds, and evaluate

Both random student split and chronological student-cohort split are reported. The chronological results are the primary deployment-oriented diagnostic, but remain limited by the retrospective label and lack of event timestamps.

In [6]:
all_results, all_predictions, all_split_rows = [], [], []
for task_name, target in TARGETS.items():
    evaluation_unit = "attempt" if task_name == "model_a_near_term_dropout" else "student_any_landmark"
    for regime in ["random", "chronological"]:
        tr_ids, va_ids, te_ids, split_meta = student_split(samples, regime=regime, seed=42)
        tr_all = samples[samples.student_id.isin(tr_ids)].copy()
        va_all = samples[samples.student_id.isin(va_ids)].copy()
        te_all = samples[samples.student_id.isin(te_ids)].copy()
        assert not (tr_ids & va_ids or tr_ids & te_ids or va_ids & te_ids)
        all_split_rows.append({"task": task_name, "evaluation_unit": evaluation_unit, "regime": regime,
                               "train_students": len(tr_ids), "validation_students": len(va_ids),
                               "test_students": len(te_ids), "train_rows": len(tr_all),
                               "validation_rows": len(va_all), "test_rows": len(te_all), **split_meta})
        for feature_set_name, selected_features in FEATURE_SETS.items():
            for model_name, model_params in MODEL_CONFIGS.items():
                feature_cols = selected_features
                model = make_model(tr_all, feature_cols, model_params)
                fit_rows = tr_all
                if model_params.get("equal_student_resample", False):
                    fit_rows = equal_student_resample(tr_all, seed=42)
                model.fit(fit_rows[feature_cols], fit_rows[target].astype(int))
                pv_rows = model.predict_proba(va_all[feature_cols])[:, 1]
                pt_rows = model.predict_proba(te_all[feature_cols])[:, 1]

                if evaluation_unit == "attempt":
                    yv = va_all[target].astype(int).to_numpy()
                    yt = te_all[target].astype(int).to_numpy()
                    pv, pt = pv_rows, pt_rows
                    val_frame = va_all[["student_id", target]].copy()
                    test_frame = te_all[["student_id", target]].copy()
                else:
                    val_frame = va_all[["student_id", target]].copy()
                    val_frame["probability"] = pv_rows
                    val_frame = val_frame.groupby("student_id", as_index=False).agg(
                        y=(target, "first"), probability=("probability", "max"))
                    test_frame = te_all[["student_id", target]].copy()
                    test_frame["probability"] = pt_rows
                    test_frame = test_frame.groupby("student_id", as_index=False).agg(
                        y=(target, "first"), probability=("probability", "max"))
                    yv, yt = val_frame["y"].astype(int).to_numpy(), test_frame["y"].astype(int).to_numpy()
                    pv, pt = val_frame["probability"].to_numpy(), test_frame["probability"].to_numpy()

                for floor in [0.40, 0.50, 0.60]:
                    threshold, feasible = choose_threshold(yv, pv, floor)
                    row = {"task": task_name, "target": target, "evaluation_unit": evaluation_unit,
                           "regime": regime, "model": model_name, "feature_set": feature_set_name,
                           "precision_floor": floor, "threshold": threshold,
                           "threshold_feasible_on_validation": feasible}
                    if feasible:
                        vm, tm = binary_metrics(yv, pv, threshold), binary_metrics(yt, pt, threshold)
                        row.update(tm)
                        row["validation_precision"], row["validation_recall"] = vm["precision"], vm["recall"]
                        row["test_precision_floor_met"] = bool(tm["precision"] >= floor)
                        all_results.append(row)
                        if evaluation_unit == "attempt":
                            pred_out = te_all[["student_id", "attempt_id", "attempt_dt", "sequence_index"]].copy()
                            pred_out["probability"] = pt_rows
                            pred_out["target"] = te_all[target].astype(int).to_numpy()
                            pred_out["predicted"] = (pt_rows >= threshold).astype(int)
                        else:
                            pred_out = test_frame.copy().rename(columns={"y": "target"})
                            pred_out["predicted"] = (pred_out["probability"] >= threshold).astype(int)
                        pred_out["task"], pred_out["evaluation_unit"], pred_out["regime"] = task_name, evaluation_unit, regime
                        pred_out["model"], pred_out["feature_set"] = model_name, feature_set_name
                        pred_out["precision_floor"], pred_out["threshold"] = floor, threshold
                        all_predictions.append(pred_out)
                    else:
                        row.update({"precision": np.nan, "recall": np.nan, "f1": np.nan, "PR_AUC": np.nan,
                                    "ROC_AUC": np.nan, "TN": np.nan, "FP": np.nan, "FN": np.nan, "TP": np.nan,
                                    "flagged": np.nan, "n": len(yv), "test_precision_floor_met": False,
                                    "note": "No validation threshold met precision floor"})
                        all_results.append(row)

results_df = pd.DataFrame(all_results)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
split_df = pd.DataFrame(all_split_rows)
display(results_df.sort_values(["regime", "task", "precision_floor", "recall"],
                               ascending=[True, True, True, False]).head(100))
display(split_df)

,task,target,evaluation_unit,regime,model,feature_set,precision_floor,threshold,threshold_feasible_on_validation,precision,...,TN,FP,FN,TP,flagged,n,validation_precision,validation_recall,test_precision_floor_met,note
30,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,mlp_16,combined_static_and_history,0.4,0.82,True,0.166667,...,255.0,5.0,9.0,1.0,6.0,270,0.500000,0.190476,False,NaN
18,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,mlp_16,temporal_history_only,0.4,0.75,True,0.000000,...,258.0,2.0,10.0,0.0,2.0,270,0.416667,0.238095,False,NaN
21,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,mlp_32_16,temporal_history_only,0.4,0.95,True,0.000000,...,260.0,0.0,10.0,0.0,0.0,270,0.500000,0.190476,False,NaN
33,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,mlp_32_16,combined_static_and_history,0.4,0.99,True,0.000000,...,260.0,0.0,10.0,0.0,0.0,270,1.000000,0.047619,False,NaN
24,model_a_near_term_dropout,target_model_a_dropout_after_next_attempt,attempt,chronological,mlp_16,student_static_only,0.4,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,281,NaN,NaN,False,No validation threshold met precision floor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,model_b_eventual_dropout,target_model_b_eventual_dropout,student_any_landmark,random,mlp_16,student_static_only,0.6,0.90,True,0.000000,...,72.0,0.0,16.0,0.0,0.0,88,1.000000,0.062500,False,NaN
50,model_b_eventual_dropout,target_model_b_eventual_dropout,student_any_landmark,random,mlp_16,combined_static_and_history,0.6,0.99,True,0.000000,...,71.0,1.0,16.0,0.0,1.0,88,1.000000,0.125000,False,NaN
41,model_b_eventual_dropout,target_model_b_eventual_dropout,student_any_landmark,random,mlp_32_16,temporal_history_only,0.6,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,88,NaN,NaN,False,No validation threshold met precision floor
47,model_b_eventual_dropout,target_model_b_eventual_dropout,student_any_landmark,random,mlp_32_16,student_static_only,0.6,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,88,NaN,NaN,False,No validation threshold met precision floor


,task,evaluation_unit,regime,train_students,validation_students,test_students,train_rows,validation_rows,test_rows,train_cutoff,validation_cutoff,test_start
0,model_a_near_term_dropout,attempt,random,262,88,88,870,305,301,NaN,NaN,NaN
1,model_a_near_term_dropout,attempt,chronological,262,88,88,925,281,270,2026-07-22,2026-08-01,2026-08-01
2,model_b_eventual_dropout,student_any_landmark,random,262,88,88,870,305,301,NaN,NaN,NaN
3,model_b_eventual_dropout,student_any_landmark,chronological,262,88,88,925,281,270,2026-07-22,2026-08-01,2026-08-01


## 6. Save outputs

Each notebook writes to its own output namespace. Existing experiment outputs are not overwritten.

In [7]:
results_df.to_csv(METRICS_OUT / "model_comparison.csv", index=False)
results_df[results_df.task.eq("model_a_near_term_dropout")].to_csv(
    METRICS_OUT / "model_a_attempt_level_metrics.csv", index=False)
results_df[results_df.task.eq("model_b_eventual_dropout")].to_csv(
    METRICS_OUT / "model_b_student_level_any_landmark_metrics.csv", index=False)
split_df.to_csv(METRICS_OUT / "student_split_report.csv", index=False)
predictions_df.to_csv(PRED_OUT / "test_predictions.csv", index=False)
print("Saved metrics to:", METRICS_OUT)
print("Saved predictions to:", PRED_OUT)

Saved metrics to: /Users/abishek/Documents/Projects/student_success_assistant/outputs/temporal_small_mlp/metrics
Saved predictions to: /Users/abishek/Documents/Projects/student_success_assistant/outputs/temporal_small_mlp/predictions


## Interpretation checklist

- Compare recall only alongside test precision and the precision-floor indicator.
- Check true positives, false negatives, false positives, and flagged counts.
- Compare history-only versus combined features.
- Treat test metrics as final diagnostics, not tuning signals.
- The any-landmark Model B score is the maximum across landmarks present in the test data; it is a retrospective aggregation, not a deployable online alert simulation.
- A positive result on the supplied dropout label does not establish prediction of a future event until event timing is defined.